# Transformer staged search -- Stage A (architecture) + Stage B (recipe) + transfer check + Stage C (candidate multiseed)

Runs the `transformer/PLAN.md` #4 staged search, **val-only** -- no test-evaluation code
path exists anywhere in this notebook. Test set03 is touched only in
`phase4_kaggle_final/04_final_loso_kaggle.ipynb`, exactly once, after the winner is
reviewed locally.

**Setup before Run All:**
1. Upload `sequences_clean/{X.npy,y.npy,meta.pkl}` (`transformer/sequences_clean/` in the
   repo) as a private Kaggle dataset named `pie-sequences-clean`, and attach it to this
   notebook.
2. Settings -> Accelerator -> **GPU T4 x2**.
3. Run All.

**Output:** `runs_search/<cfg_id>/seed<k>.json` per config (val-only schema -- no `test`
key, ever) + `runs_search/_stage_summary.json`, zipped to
`/kaggle/working/transformer_search_output.zip`. Download it, unzip into
`transformer/phase2_kaggle_search/` so `runs_search/` sits there, then run
`transformer/phase3_search_review/03_search_report.py` locally -- that is the Phase-T3
human checkpoint, before the winner config is pasted into
`phase4_kaggle_final/04_final_loso_kaggle.ipynb`.

Nothing here should diverge from `transformer/PLAN.md` sections 3-4 (the grids, the
`transformer_default` config, the selection rules) -- if it does, PLAN.md is stale and
should be corrected, not this notebook.

In [ ]:
import json, os, subprocess, sys, tempfile, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

print("python:", sys.version)
print("torch:", torch.__version__)
n_gpu = torch.cuda.device_count()
print("CUDA available:", torch.cuda.is_available(), "| device count:", n_gpu)
for i in range(n_gpu):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}")
if n_gpu != 2:
    print(f"WARNING: expected 2x T4 (transformer/PLAN.md #5), found {n_gpu} GPU(s). "
          f"Continuing with {max(n_gpu, 1)} worker(s) -- this only affects wall-clock, "
          f"not correctness.")

GPU_IDS = list(range(n_gpu)) if n_gpu > 0 else [None]  # None => CPU fallback (slow, not expected)
print("worker GPU ids:", GPU_IDS)

In [ ]:
candidates = [Path("/kaggle/input/pie-sequences-clean")]
candidates += sorted(Path("/kaggle/input").glob("*"))
SEQ_DIR = None
for c in candidates:
    if (c / "X.npy").exists() and (c / "y.npy").exists() and (c / "meta.pkl").exists():
        SEQ_DIR = c
        break
assert SEQ_DIR is not None, (
    "Could not find sequences_clean/{X.npy,y.npy,meta.pkl} under /kaggle/input/. "
    "Attach the 'pie-sequences-clean' dataset (see the markdown cell above)."
)
print("using sequences from:", SEQ_DIR)

X_check = np.load(SEQ_DIR / "X.npy", mmap_mode="r")
assert X_check.shape == (4906, 16, 5), (
    f"unexpected X shape {X_check.shape} -- this must be the CLEAN sequences_clean/ "
    f"(N=4906), not the old leaky sequences/ (N=1389). Re-upload the correct dataset."
)
print(f"asserted clean data: X.shape == {X_check.shape}")

HERE_DIR = Path("/kaggle/working")
OUT_ROOT = HERE_DIR / "runs_search"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
SEEDS = [42, 0, 1, 2, 3]
print("out_root:", OUT_ROOT)

In [ ]:
# KEEP IN SYNC WITH transformer/00_transformer_model.py -- edit there first, then
# regenerate this notebook (see gen_search_nb.py). Do not hand-edit this cell.
model_code = '"""00_transformer_model.py — TransformerIntentPredictor.\n\nSingle source of truth for the transformer architecture (transformer/PLAN.md #3).\nThe Kaggle notebooks (phase2_kaggle_search/03_search_kaggle.ipynb,\nphase4_kaggle_final/04_final_loso_kaggle.ipynb) embed this class verbatim under a\n"KEEP IN SYNC WITH 00_transformer_model.py" banner — edit here first, then copy down.\n\nForward contract matches pipeline/03_bilstm_model.py::BiLSTMIntentPredictor:\n    input  (B, 16, 5) float32, already z-scored by the caller\n    output (B, 1) float32 raw logits (sigmoid applied by the caller)\nNo required constructor args (defaults = transformer_default, PLAN.md #3).\n"""\nimport math\n\nimport torch\nimport torch.nn as nn\n\n\ndef _sinusoidal_pe(length, d_model):\n    pe = torch.zeros(length, d_model)\n    position = torch.arange(length, dtype=torch.float32).unsqueeze(1)\n    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32)\n                         * (-math.log(10000.0) / d_model))\n    pe[:, 0::2] = torch.sin(position * div_term)\n    pe[:, 1::2] = torch.cos(position * div_term[: pe[:, 1::2].shape[1]])\n    return pe.unsqueeze(0)\n\n\nclass TransformerIntentPredictor(nn.Module):\n    def __init__(self, input_dim=5, d_model=128, nhead=4, num_layers=2, dim_ff=None,\n                 dropout=0.1, pool="cls", pos="learned", seq_len=16, norm_first=True):\n        super().__init__()\n        if d_model % nhead != 0:\n            raise ValueError(f"d_model={d_model} must be divisible by nhead={nhead}")\n        if pool not in ("cls", "mean", "last"):\n            raise ValueError(f"pool must be one of cls/mean/last, got {pool!r}")\n        if pos not in ("learned", "sin"):\n            raise ValueError(f"pos must be one of learned/sin, got {pos!r}")\n\n        dim_ff = dim_ff or 2 * d_model\n        self.pool = pool\n        self.use_cls = pool == "cls"\n        self.seq_len = seq_len\n\n        self.input_proj = nn.Linear(input_dim, d_model)\n\n        pe_len = seq_len + 1 if self.use_cls else seq_len\n        if self.use_cls:\n            self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)\n        if pos == "learned":\n            self.pos_embed = nn.Parameter(torch.randn(1, pe_len, d_model) * 0.02)\n        else:\n            self.register_buffer("pos_embed", _sinusoidal_pe(pe_len, d_model), persistent=False)\n\n        self.embed_drop = nn.Dropout(dropout)\n\n        if num_layers > 0:\n            layer = nn.TransformerEncoderLayer(\n                d_model, nhead, dim_feedforward=dim_ff, dropout=dropout,\n                activation="gelu", batch_first=True, norm_first=norm_first,\n            )\n            self.encoder = nn.TransformerEncoder(\n                layer, num_layers=num_layers, norm=nn.LayerNorm(d_model),\n                enable_nested_tensor=False,\n            )\n        else:\n            # Gate-1 linear-probe floor: skip the encoder entirely (PLAN.md Phase T1 Gate 1).\n            self.encoder = None\n\n        self.head_drop = nn.Dropout(dropout)\n        self.head = nn.Linear(d_model, 1)\n\n    def forward(self, x):\n        B = x.shape[0]\n        h = self.input_proj(x)\n        if self.use_cls:\n            cls = self.cls_token.expand(B, -1, -1)\n            h = torch.cat([cls, h], dim=1)\n        h = h + self.pos_embed[:, : h.shape[1], :]\n        h = self.embed_drop(h)\n        if self.encoder is not None:\n            h = self.encoder(h)\n\n        if self.pool == "cls":\n            pooled = h[:, 0, :]\n        elif self.pool == "last":\n            pooled = h[:, -1, :]\n        else:  # mean — exclude the CLS token if present\n            seq = h[:, 1:, :] if self.use_cls else h\n            pooled = seq.mean(dim=1)\n\n        pooled = self.head_drop(pooled)\n        return self.head(pooled)\n\n\ndef count_params(model):\n    return sum(p.numel() for p in model.parameters())\n\n\nif __name__ == "__main__":\n    torch.manual_seed(0)\n    x = torch.randn(4, 16, 5)\n\n    print("Shape self-test (all pool x pos combos):")\n    for pool in ("cls", "mean", "last"):\n        for pos in ("learned", "sin"):\n            m = TransformerIntentPredictor(pool=pool, pos=pos)\n            out = m(x)\n            assert out.shape == (4, 1), f"bad shape {out.shape} for pool={pool} pos={pos}"\n    print("  OK — every combo produces (B,1) logits")\n\n    print("\\nParameter ladder (Stage-A sizes; brackets the 594,561-param BiLSTM):")\n    print(f"{\'(d_model, ff)\':>16} {\'L\':>3} {\'params\':>10}")\n    for d_model, ff in [(64, 128), (128, 256), (128, 512)]:\n        for L in (2, 4):\n            n = count_params(TransformerIntentPredictor(d_model=d_model, dim_ff=ff, num_layers=L))\n            print(f"{f\'({d_model}, {ff})\':>16} {L:>3} {n:>10,}")\n\n    default = TransformerIntentPredictor(d_model=128, num_layers=2, dim_ff=256, dropout=0.1,\n                                         pool="cls", pos="learned")\n    print(f"\\ntransformer_default params: {count_params(default):,}  (BiLSTM baseline: 594,561)")\n\n    floor = TransformerIntentPredictor(num_layers=0, pool="mean")\n    assert floor(x).shape == (4, 1)\n    print(f"L=0 linear-probe floor params: {count_params(floor):,}")\n'
(HERE_DIR / "00_transformer_model.py").write_text(model_code)
print("wrote", HERE_DIR / "00_transformer_model.py", f"({len(model_code)} bytes)")

In [ ]:
# KEEP IN SYNC WITH transformer/02_train_transformer.py -- edit there first, then
# regenerate this notebook (see gen_search_nb.py). Do not hand-edit this cell.
engine_code = '"""02_train_transformer.py — the transformer training engine (transformer/PLAN.md #2, #4).\n\ntrain_run() is the single training loop reused everywhere: Phase-T1 dev runs here,\nthe Kaggle search notebook\'s Stages A/B/C, and the Kaggle final notebook\'s Stage D. The\nnotebooks embed this file verbatim under a "KEEP IN SYNC WITH 02_train_transformer.py"\nbanner — edit here first, then copy down.\n\nFrozen protocol (identical to the LSTM — pipeline/04_train_bilstm.py /\njournal_prep/issue2_clean_protocol/06b_local_verify_seed42.py): sequences_clean/,\nsplits by recording set, train-only z-score, pos_weight=1.682, threshold 0.5, batch 32,\nmax 100 epochs, early-stop patience 15 on val AUC, best-on-val-AUC checkpoint,\nseeds [42,0,1,2,3]. Only the model architecture and training RECIPE (lr / schedule /\ndropout / weight decay / optimizer) are searchable — see PLAN.md #4 for the grids.\n\nCLI (single dev run — NOT a search; --eval-test only belongs in Phase T4 / Stage D):\n    python transformer/phase1_setup/02_train_transformer.py --preset default --seed 42 \\\n        --device cpu --out_dir transformer/phase1_setup/runs_dev/default_seed42\n"""\nimport argparse\nimport importlib.util\nimport json\nimport math\nimport pickle\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom sklearn.metrics import (accuracy_score, average_precision_score, f1_score,\n                             precision_score, recall_score, roc_auc_score)\nfrom torch.utils.data import DataLoader, TensorDataset\n\nHERE = Path(__file__).resolve().parent\nSEQ_DIR = HERE.parent / "sequences_clean"  # shared data lives at transformer/ root\n\n_spec = importlib.util.spec_from_file_location("transformer_model", HERE / "00_transformer_model.py")\n_m = importlib.util.module_from_spec(_spec)\n_spec.loader.exec_module(_m)\nTransformerIntentPredictor = _m.TransformerIntentPredictor\ncount_params = _m.count_params\n\nTRAIN_SETS = {"set01", "set02", "set04"}\nVAL_SETS = {"set05", "set06"}\nTEST_SETS = {"set03"}\nPOS_WEIGHT = 1.682\nEPOCHS = 100\nBATCH_SIZE = 32\nPLATEAU_PATIENCE = 5          # ReduceLROnPlateau patience (LSTM parity)\nEARLY_STOP_PATIENCE = 15\nTHRESHOLD = 0.5\nWARMUP_EPOCHS = 5\nCOSINE_MIN_FRAC = 0.01\n\n# The pre-registered "transformer_default" (PLAN.md #3): the transformer\'s own\n# architecture, trained with the LSTM\'s exact recipe (Adam 1e-3 / wd 1e-5 / plateau).\n# Zero search — fixed before any search run.\nDEFAULT_CFG = dict(\n    d_model=128, nhead=4, num_layers=2, dim_ff=256, dropout=0.1, pool="cls", pos="learned",\n    lr=1e-3, schedule="plateau", weight_decay=1e-5, optimizer="adam",\n)\n\nPRESETS = {"default": DEFAULT_CFG}\n\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cudnn.benchmark = False\n\n\ndef load_raw(seq_dir=SEQ_DIR):\n    X = np.load(Path(seq_dir) / "X.npy").astype(np.float32)\n    y = np.load(Path(seq_dir) / "y.npy").astype(np.float32)\n    with open(Path(seq_dir) / "meta.pkl", "rb") as f:\n        meta = pickle.load(f)\n    set_ids = np.array([m["set_id"] for m in meta])\n    assert X.shape == (4906, 16, 5), f"unexpected X shape {X.shape} — wrong sequences dir?"\n    return X, y, set_ids\n\n\ndef split_data(X, y, set_ids):\n    tr = np.isin(set_ids, list(TRAIN_SETS))\n    va = np.isin(set_ids, list(VAL_SETS))\n    te = np.isin(set_ids, list(TEST_SETS))\n    return X[tr], y[tr], X[va], y[va], X[te], y[te]\n\n\ndef build_model(cfg):\n    return TransformerIntentPredictor(\n        input_dim=5, d_model=cfg["d_model"], nhead=cfg.get("nhead", 4),\n        num_layers=cfg["num_layers"], dim_ff=cfg["dim_ff"], dropout=cfg["dropout"],\n        pool=cfg["pool"], pos=cfg["pos"],\n    )\n\n\ndef build_optimizer(model, cfg):\n    lr, wd = cfg["lr"], cfg["weight_decay"]\n    if cfg.get("optimizer", "adam") == "adamw":\n        no_decay_keys = ("bias", "norm", "pos_embed", "cls_token")\n        decay_params, no_decay_params = [], []\n        for name, p in model.named_parameters():\n            if not p.requires_grad:\n                continue\n            target = no_decay_params if any(k in name.lower() for k in no_decay_keys) else decay_params\n            target.append(p)\n        groups = [{"params": decay_params, "weight_decay": wd},\n                  {"params": no_decay_params, "weight_decay": 0.0}]\n        return torch.optim.AdamW(groups, lr=lr)\n    return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)\n\n\ndef lr_for_epoch(epoch, base_lr):\n    """warmup_cosine schedule only; \'plateau\' is stepped via ReduceLROnPlateau instead."""\n    if epoch <= WARMUP_EPOCHS:\n        return base_lr * epoch / WARMUP_EPOCHS\n    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)\n    cosine = 0.5 * (1 + math.cos(math.pi * progress))\n    return base_lr * (COSINE_MIN_FRAC + (1 - COSINE_MIN_FRAC) * cosine)\n\n\n@torch.no_grad()\ndef evaluate(model, loader, device):\n    model.eval()\n    crit = nn.BCEWithLogitsLoss(reduction="sum")\n    probs_all, labels_all, total, n = [], [], 0.0, 0\n    for xb, yb in loader:\n        xb, yb = xb.to(device), yb.to(device)\n        logits = model(xb).squeeze(-1)\n        total += crit(logits, yb).item()\n        probs_all.append(torch.sigmoid(logits).cpu().numpy())\n        labels_all.append(yb.cpu().numpy())\n        n += yb.size(0)\n    probs = np.concatenate(probs_all)\n    labels = np.concatenate(labels_all)\n    preds = (probs >= THRESHOLD).astype(int)\n    metrics = {\n        "loss": total / n,\n        "acc": accuracy_score(labels, preds),\n        "f1": f1_score(labels, preds, zero_division=0),\n        "auc": roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else float("nan"),\n        "pr_auc": average_precision_score(labels, probs),\n        "prec": precision_score(labels, preds, zero_division=0),\n        "rec": recall_score(labels, preds, zero_division=0),\n    }\n    return metrics, probs, labels\n\n\ndef make_loader(X, y, shuffle):\n    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))\n    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0, pin_memory=False)\n\n\ndef train_run(cfg, seed, device, data, eval_test=False, out_dir=None, verbose=False,\n              pos_weight=None):\n    """The training loop shared by every stage. `data` = (Xtr, ytr, Xva, yva, Xte, yte),\n    RAW (unnormalized) arrays for the fixed set-based split (identical across seeds).\n    eval_test=True touches the test set — only Stage D (Phase T4) may pass this.\n    pos_weight overrides the frozen POS_WEIGHT=1.682 — only Phase T4\'s LOSO folds pass\n    this (Issue-5 protocol: per-fold n_neg/n_pos, since each fold\'s train pool differs)."""\n    set_seed(seed)\n    Xtr, ytr, Xva, yva, Xte, yte = data\n    pw = POS_WEIGHT if pos_weight is None else pos_weight\n\n    flat = Xtr.reshape(-1, Xtr.shape[-1])\n    mean, std = flat.mean(axis=0), flat.std(axis=0) + 1e-6\n    Xtr_n = (Xtr - mean) / std\n    Xva_n = (Xva - mean) / std\n    Xte_n = (Xte - mean) / std\n\n    train_loader = make_loader(Xtr_n, ytr, shuffle=True)\n    val_loader = make_loader(Xva_n, yva, shuffle=False)\n    test_loader = make_loader(Xte_n, yte, shuffle=False) if eval_test else None\n\n    model = build_model(cfg).to(device)\n    n_params = count_params(model)\n    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], device=device))\n    opt = build_optimizer(model, cfg)\n    schedule = cfg.get("schedule", "plateau")\n    plateau = None\n    if schedule == "plateau":\n        plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(\n            opt, mode="max", factor=0.5, patience=PLATEAU_PATIENCE)\n\n    best_auc, best_epoch, best_state, no_improve = -1.0, 0, None, 0\n    history = []\n    t0 = time.time()\n    for epoch in range(1, EPOCHS + 1):\n        if schedule == "warmup_cosine":\n            new_lr = lr_for_epoch(epoch, cfg["lr"])\n            for g in opt.param_groups:\n                g["lr"] = new_lr\n\n        model.train()\n        for xb, yb in train_loader:\n            xb, yb = xb.to(device), yb.to(device)\n            opt.zero_grad()\n            loss = crit(model(xb).squeeze(-1), yb)\n            loss.backward()\n            opt.step()\n\n        val_metrics, _, _ = evaluate(model, val_loader, device)\n        if schedule == "plateau":\n            plateau.step(val_metrics["auc"])\n        history.append({"epoch": epoch, **{f"val_{k}": v for k, v in val_metrics.items()}})\n        if verbose:\n            print(f"  ep {epoch:3d} vAUC {val_metrics[\'auc\']:.4f}")\n\n        if val_metrics["auc"] > best_auc:\n            best_auc, best_epoch = val_metrics["auc"], epoch\n            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}\n            no_improve = 0\n        else:\n            no_improve += 1\n            if no_improve >= EARLY_STOP_PATIENCE:\n                break\n\n    model.load_state_dict(best_state)\n    final_val_metrics, _, _ = evaluate(model, val_loader, device)\n    seconds = round(time.time() - t0, 1)\n\n    result = dict(**cfg, seed=seed, n_params=n_params, best_epoch=best_epoch,\n                  val=final_val_metrics, seconds=seconds, pos_weight=pw)\n\n    if eval_test:\n        test_metrics, test_probs, test_labels = evaluate(model, test_loader, device)\n        preds = (test_probs >= THRESHOLD).astype(int)\n        tn = int(((test_labels == 0) & (preds == 0)).sum())\n        fp = int(((test_labels == 0) & (preds == 1)).sum())\n        fn = int(((test_labels == 1) & (preds == 0)).sum())\n        tp = int(((test_labels == 1) & (preds == 1)).sum())\n        result["test"] = test_metrics\n        result["test_confusion_matrix"] = [[tn, fp], [fn, tp]]\n\n    if out_dir is not None:\n        out_dir = Path(out_dir)\n        out_dir.mkdir(parents=True, exist_ok=True)\n        torch.save({"model": best_state, "epoch": best_epoch, "val_metrics": final_val_metrics},\n                   out_dir / "best.pt")\n        np.save(out_dir / "norm_mean.npy", mean)\n        np.save(out_dir / "norm_std.npy", std)\n        (out_dir / "history.json").write_text(json.dumps(history, indent=2))\n        final_json = {"best_epoch": best_epoch, "val": final_val_metrics}\n        if eval_test:\n            final_json["test"] = result["test"]\n            final_json["test_confusion_matrix"] = result["test_confusion_matrix"]\n        (out_dir / "final.json").write_text(json.dumps(final_json, indent=2))\n\n    return result\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--preset", default="default", choices=list(PRESETS.keys()))\n    ap.add_argument("--seed", type=int, default=42)\n    ap.add_argument("--device", default="cpu", choices=["cpu", "cuda", "mps"])\n    ap.add_argument("--seq_dir", default=str(SEQ_DIR))\n    ap.add_argument("--out_dir", default=None)\n    ap.add_argument("--eval-test", action="store_true",\n                    help="Touch the test set. Only pass this in Phase T4 (Stage D).")\n    ap.add_argument("--verbose", action="store_true")\n    args = ap.parse_args()\n\n    device = torch.device(args.device)\n    cfg = PRESETS[args.preset]\n    X, y, set_ids = load_raw(Path(args.seq_dir))\n    data = split_data(X, y, set_ids)\n\n    if args.eval_test:\n        print("*** --eval-test set: this run touches set03. Only do this once, in Phase T4. ***")\n\n    result = train_run(cfg, args.seed, device, data, eval_test=args.eval_test,\n                       out_dir=args.out_dir, verbose=args.verbose)\n    printable = {k: v for k, v in result.items() if k != "val"}\n    print(json.dumps(printable, indent=2, default=str))\n    print("val:", result["val"])\n    if "test" in result:\n        print("test:", result["test"])\n\n\nif __name__ == "__main__":\n    main()\n'
(HERE_DIR / "02_train_transformer.py").write_text(engine_code)
print("wrote", HERE_DIR / "02_train_transformer.py", f"({len(engine_code)} bytes)")

import importlib.util
_spec_engine = importlib.util.spec_from_file_location(
    "train_engine", HERE_DIR / "02_train_transformer.py")
engine = importlib.util.module_from_spec(_spec_engine)
_spec_engine.loader.exec_module(engine)
print("loaded engine module; DEFAULT_CFG =", engine.DEFAULT_CFG)

In [ ]:
# Auto-generated worker script -- see gen_search_nb.py's WORKER_CODE.
worker_code = '"""worker.py -- one GPU-pinned training worker for the Kaggle search notebook.\nAuto-generated by phase2_kaggle_search/03_search_kaggle.ipynb -- edit the notebook cell,\nnot this file.\nCUDA_VISIBLE_DEVICES is set by the parent process before this script starts, so\ntorch.device("cuda") always refers to this worker\'s assigned physical GPU.\n"""\nimport argparse\nimport importlib.util\nimport json\nfrom pathlib import Path\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--jobs", required=True)\n    ap.add_argument("--seq_dir", required=True)\n    ap.add_argument("--out_root", required=True)\n    ap.add_argument("--here", required=True)\n    args = ap.parse_args()\n\n    here = Path(args.here)\n    spec = importlib.util.spec_from_file_location("train_engine", here / "02_train_transformer.py")\n    engine = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(engine)\n\n    import torch\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n    X, y, set_ids = engine.load_raw(Path(args.seq_dir))\n    data = engine.split_data(X, y, set_ids)\n\n    jobs = json.loads(Path(args.jobs).read_text())\n    out_root = Path(args.out_root)\n\n    for job in jobs:\n        assert not job["eval_test"], (\n            "search notebook must never evaluate the test set -- "\n            f"got eval_test=True for {job[\'cfg_id\']} seed{job[\'seed\']}"\n        )\n        out_path = out_root / job["cfg_id"] / f"seed{job[\'seed\']}.json"\n        if out_path.exists():\n            print(f"[worker] cached {job[\'cfg_id\']} seed{job[\'seed\']}", flush=True)\n            continue\n        out_path.parent.mkdir(parents=True, exist_ok=True)\n        result = engine.train_run(job["cfg"], job["seed"], device, data,\n                                  eval_test=False, out_dir=None)\n        out_path.write_text(json.dumps(result, indent=2, default=str))\n        print(f"[worker] done {job[\'cfg_id\']} seed{job[\'seed\']} "\n              f"valAUC={result[\'val\'][\'auc\']:.4f} ({result[\'seconds\']}s)", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
(HERE_DIR / "worker.py").write_text(worker_code)
print("wrote", HERE_DIR / "worker.py", f"({len(worker_code)} bytes)")

In [ ]:
# --- config identity, grids, and the 2-GPU subprocess runner ---
#
# cfg_id() MUST encode every field that changes training behaviour, including
# `optimizer` -- Stage A trains with Adam (the frozen default recipe) and Stage B
# always uses AdamW, so a Stage-B cell can numerically match Stage A's default recipe
# on every OTHER field (lr=1e-3, schedule=plateau, dropout=0.1, wd=1e-5) while differing
# only in optimizer. Omitting `optimizer` from the id would collide the two into the
# same cache path and silently discard one run's result.

ARCH_KEYS = ["d_model", "nhead", "num_layers", "dim_ff", "pool", "pos"]
RECIPE_KEYS = ["lr", "schedule", "dropout", "weight_decay", "optimizer"]

ARCH_D_FF = [(64, 128), (128, 256), (128, 512)]
ARCH_LAYERS = [2, 4]
ARCH_POOL = ["cls", "mean", "last"]
ARCH_POS = ["learned", "sin"]

RECIPE_LR = [1e-4, 3e-4, 1e-3]
RECIPE_SCHEDULE = ["plateau", "warmup_cosine"]
RECIPE_DROPOUT = [0.1, 0.3, 0.5]
RECIPE_WD = [1e-5, 1e-2]


def pos_abbrev(pos):
    return "lpe" if pos == "learned" else "spe"


def cfg_id(cfg):
    arch = f"d{cfg['d_model']}_ff{cfg['dim_ff']}_L{cfg['num_layers']}_{cfg['pool']}_{pos_abbrev(cfg['pos'])}"
    recipe = (f"{cfg['optimizer']}_lr{cfg['lr']:.0e}_{cfg['schedule']}_"
             f"do{cfg['dropout']}_wd{cfg['weight_decay']:.0e}")
    return f"{arch}__{recipe}"


def arch_only(cfg):
    return {k: cfg[k] for k in ARCH_KEYS}


def recipe_only(cfg):
    return {k: cfg[k] for k in RECIPE_KEYS}


DEFAULT_CFG = engine.DEFAULT_CFG
DEFAULT_RECIPE = recipe_only(DEFAULT_CFG)
print("DEFAULT_RECIPE:", DEFAULT_RECIPE, "\ndefault cfg_id:", cfg_id(DEFAULT_CFG))


def build_stage_a_grid(recipe=None):
    recipe = recipe or DEFAULT_RECIPE
    grid = []
    for d_model, ff in ARCH_D_FF:
        for L in ARCH_LAYERS:
            for pool in ARCH_POOL:
                for pos in ARCH_POS:
                    grid.append(dict(d_model=d_model, nhead=4, num_layers=L, dim_ff=ff,
                                     pool=pool, pos=pos, **recipe))
    return grid  # 3*2*3*2 = 36


def build_stage_b_grid(arch_cfg):
    base_arch = arch_only(arch_cfg)
    grid = []
    for lr in RECIPE_LR:
        for schedule in RECIPE_SCHEDULE:
            for dropout in RECIPE_DROPOUT:
                for wd in RECIPE_WD:
                    grid.append(dict(**base_arch, lr=lr, schedule=schedule, dropout=dropout,
                                     weight_decay=wd, optimizer="adamw"))
    return grid  # 3*2*3*2 = 36


def build_transfer_grid(recipe_cfgs, arch_cfgs):
    grid = []
    for r in recipe_cfgs:
        for a in arch_cfgs:
            grid.append(dict(**arch_only(a), **recipe_only(r)))
    return grid  # len(recipe_cfgs) * len(arch_cfgs)


def make_jobs(cfgs, seed):
    return [dict(cfg=c, seed=seed, eval_test=False, cfg_id=cfg_id(c)) for c in cfgs]


def make_jobs_multiseed(cfgs, seeds):
    return [dict(cfg=c, seed=s, eval_test=False, cfg_id=cfg_id(c)) for c in cfgs for s in seeds]


def run_stage_parallel(jobs, gpu_ids, label=""):
    shards = [[] for _ in gpu_ids]
    for i, job in enumerate(jobs):
        shards[i % len(gpu_ids)].append(job)

    procs, shard_files = [], []
    t0 = time.time()
    for gpu, shard in zip(gpu_ids, shards):
        if not shard:
            continue
        f = tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False, dir=str(HERE_DIR))
        json.dump(shard, f)
        f.close()
        shard_files.append(f.name)
        env = dict(os.environ)
        if gpu is not None:
            env["CUDA_VISIBLE_DEVICES"] = str(gpu)
        proc = subprocess.Popen(
            [sys.executable, str(HERE_DIR / "worker.py"),
             "--jobs", f.name, "--seq_dir", str(SEQ_DIR),
             "--out_root", str(OUT_ROOT), "--here", str(HERE_DIR)],
            env=env,
        )
        procs.append(proc)
        print(f"[{label}] launched worker on gpu={gpu} with {len(shard)} job(s)")

    for p in procs:
        ret = p.wait()
        if ret != 0:
            raise RuntimeError(f"[{label}] worker exited with code {ret}")
    for fn in shard_files:
        os.unlink(fn)
    print(f"[{label}] all workers done in {time.time() - t0:.0f}s")


def load_stage_results(jobs):
    results = []
    for job in jobs:
        p = OUT_ROOT / job["cfg_id"] / f"seed{job['seed']}.json"
        if not p.exists():
            raise FileNotFoundError(f"missing result: {p}")
        results.append(json.loads(p.read_text()))
    return results


def rank_by_val_auc(results):
    return sorted(results, key=lambda r: r["val"]["auc"], reverse=True)

In [ ]:
print("=== Stage A: architecture grid (36 configs, seed 42, val-only) ===")
stage_a_grid = build_stage_a_grid()
stage_a_jobs = make_jobs(stage_a_grid, seed=42)
assert len(stage_a_jobs) == 36, len(stage_a_jobs)
run_stage_parallel(stage_a_jobs, GPU_IDS, label="Stage A")
stage_a_results = load_stage_results(stage_a_jobs)

n_diverged = sum(1 for r in stage_a_results if r["val"]["auc"] < 0.70)
if n_diverged > len(stage_a_results) // 3:
    print(f"CONTINGENCY (PLAN.md #4): {n_diverged}/{len(stage_a_results)} cells diverged "
          f"(val AUC < 0.70) at lr=1e-3 -- rerunning Stage A once at lr=3e-4.")
    recipe_lo = dict(DEFAULT_RECIPE)
    recipe_lo["lr"] = 3e-4
    stage_a_grid = build_stage_a_grid(recipe=recipe_lo)
    stage_a_jobs = make_jobs(stage_a_grid, seed=42)
    run_stage_parallel(stage_a_jobs, GPU_IDS, label="Stage A (lr=3e-4 amendment)")
    stage_a_results = load_stage_results(stage_a_jobs)

stage_a_ranked = rank_by_val_auc(stage_a_results)
df_a = pd.DataFrame([
    {**{k: r[k] for k in ["d_model", "dim_ff", "num_layers", "pool", "pos"]},
     "val_auc": r["val"]["auc"], "n_params": r["n_params"]}
    for r in stage_a_ranked
])
print(df_a.head(10).to_string(index=False))

arch_top3 = [stage_a_ranked[i] for i in range(3)]
print("\nStage-A top-3 architectures:")
for i, r in enumerate(arch_top3, 1):
    print(f"  #{i}: {cfg_id(r)}  val_auc={r['val']['auc']:.4f}")

In [ ]:
print("\n=== Stage B: training-recipe grid on Stage-A #1 (36 configs, seed 42, val-only) ===")
stage_b_grid = build_stage_b_grid(arch_top3[0])
stage_b_jobs = make_jobs(stage_b_grid, seed=42)
assert len(stage_b_jobs) == 36, len(stage_b_jobs)
run_stage_parallel(stage_b_jobs, GPU_IDS, label="Stage B")
stage_b_results = load_stage_results(stage_b_jobs)
stage_b_ranked = rank_by_val_auc(stage_b_results)

df_b = pd.DataFrame([
    {**{k: r[k] for k in ["lr", "schedule", "dropout", "weight_decay", "optimizer"]},
     "val_auc": r["val"]["auc"]}
    for r in stage_b_ranked
])
print(df_b.head(10).to_string(index=False))

recipe_top3 = [stage_b_ranked[i] for i in range(3)]
print("\nStage-B top-3 recipes (on architecture #1):")
for i, r in enumerate(recipe_top3, 1):
    print(f"  #{i}: {cfg_id(r)}  val_auc={r['val']['auc']:.4f}")

In [ ]:
print("\n=== Transfer check: top-3 recipes x Stage-A architectures #2, #3 (6 configs, seed 42) ===")
transfer_grid = build_transfer_grid(recipe_top3, [arch_top3[1], arch_top3[2]])
transfer_jobs = make_jobs(transfer_grid, seed=42)
assert len(transfer_jobs) == 6, len(transfer_jobs)
run_stage_parallel(transfer_jobs, GPU_IDS, label="Transfer check")
transfer_results = load_stage_results(transfer_jobs)
for r in rank_by_val_auc(transfer_results):
    print(f"  {cfg_id(r)}  val_auc={r['val']['auc']:.4f}")

In [ ]:
print("\n=== Stage C: candidate multiseed (top-5 pooled + transformer_default, 5 seeds, val-only) ===")
pool = stage_a_results + stage_b_results + transfer_results
pool_ranked = rank_by_val_auc(pool)

seen_ids, top5_cfgs = set(), []
for r in pool_ranked:
    cid = cfg_id(r)
    if cid in seen_ids:
        continue
    seen_ids.add(cid)
    top5_cfgs.append({k: r[k] for k in ARCH_KEYS + RECIPE_KEYS})
    if len(top5_cfgs) == 5:
        break

candidates = list(top5_cfgs)
if cfg_id(DEFAULT_CFG) not in {cfg_id(c) for c in candidates}:
    candidates.append(dict(DEFAULT_CFG))

print(f"Stage-C candidates ({len(candidates)}):")
for c in candidates:
    tag = "  <- transformer_default" if cfg_id(c) == cfg_id(DEFAULT_CFG) else ""
    print(f"  {cfg_id(c)}{tag}")

stage_c_jobs = make_jobs_multiseed(candidates, SEEDS)
run_stage_parallel(stage_c_jobs, GPU_IDS, label="Stage C")
stage_c_results = load_stage_results(stage_c_jobs)

rows = []
for c in candidates:
    cid = cfg_id(c)
    vals = [r["val"]["auc"] for r in stage_c_results if cfg_id(r) == cid]
    rows.append({"cfg_id": cid, "n_seeds": len(vals),
                 "val_auc_mean": float(np.mean(vals)),
                 "val_auc_std": float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0})
df_c = pd.DataFrame(rows).sort_values("val_auc_mean", ascending=False)
print(df_c.to_string(index=False))

winner_cid = df_c.iloc[0]["cfg_id"]
winner_cfg = next(c for c in candidates if cfg_id(c) == winner_cid)
is_default = winner_cid == cfg_id(DEFAULT_CFG)
print(f"\nWINNER (highest mean val AUC): {winner_cid}"
     f"{'  (== transformer_default)' if is_default else ''}")

In [ ]:
summary = {
    "stage_a_top3": [{"cfg": {k: r[k] for k in ARCH_KEYS + RECIPE_KEYS}, "val_auc": r["val"]["auc"]}
                     for r in arch_top3],
    "stage_b_top3": [{"cfg": {k: r[k] for k in ARCH_KEYS + RECIPE_KEYS}, "val_auc": r["val"]["auc"]}
                     for r in recipe_top3],
    "stage_c_candidates": rows,
    "winner_cfg_id": winner_cid,
    "winner_cfg": winner_cfg,
    "winner_is_default": is_default,
    "default_cfg_id": cfg_id(DEFAULT_CFG),
    "default_cfg": DEFAULT_CFG,
    "seeds": SEEDS,
}
(OUT_ROOT / "_stage_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("wrote", OUT_ROOT / "_stage_summary.json")

import shutil
zip_path = Path("/kaggle/working/transformer_search_output.zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=str(OUT_ROOT))
print(f"\nDONE. Download {zip_path} and unzip its contents into "
     f"transformer/phase2_kaggle_search/ (so runs_search/ sits there), then run "
     f"transformer/phase3_search_review/03_search_report.py locally "
     f"(Phase T3 -- review the winner BEFORE phase4_kaggle_final/ ever touches test).")

## Next steps

1. Download `/kaggle/working/transformer_search_output.zip`.
2. Unzip into `transformer/phase2_kaggle_search/` so `runs_search/` sits at
   `transformer/phase2_kaggle_search/runs_search/`.
3. Locally: `python transformer/phase3_search_review/03_search_report.py` -- re-derives
   the ranking tables/figure and prints the winner. **Review it (Phase T3) before
   proceeding** -- test set03 still does not exist anywhere yet.
4. Paste the winner config into `phase4_kaggle_final/04_final_loso_kaggle.ipynb`'s
   CONFIG cell and run it -- that notebook touches test exactly once (Stage D) and runs
   the 6-fold LOSO.